# Telco Customer Churn Analysis
## Notebook 1 — Data Cleaning & Feature Engineering

**Dataset:** IBM Watson Analytics — Telco Customer Churn (via Kaggle)  
**Tools:** Python · Pandas · NumPy  

## 1. Imports & Configuration

In [19]:
#Importing required libraries
import pandas as pd
import numpy as np
import warnings

#Ignore warning
warnings.filterwarnings(action = 'ignore')

#Display settings - show all columns without truncation
pd.set_option('display.max_columns', None)   # Show every column
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 decimal places for floats

print('Libraries loaded successfully')

Libraries loaded successfully


## 2. Load the Raw Dataset

In [20]:
df = pd.read_csv('data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f'Rows:{df.shape[0]:,}') #number of rows
print(f'Columns:{df.shape[1]}') #number of columns
print(f'\nColumn Names:') 
print(list(df.columns))

Rows:7,043
Columns:21

Column Names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [21]:
# Preview first 5 rows
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [22]:
# Check data types and null count of every column
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [23]:
# Statistical summary of numeric columns
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.00,7043.00,7043.00
mean,0.16,32.37,64.76
std,0.37,24.56,30.09
min,0.00,0.00,18.25
25%,0.00,9.00,35.50
50%,0.00,29.00,70.35
75%,0.00,55.00,89.85
max,1.00,72.00,118.75


## 3. Churn Distribution (Target Column)

In [24]:
# TARGET VARIABLE VALIDATION — data quality check
print('Churn Column :\n')
print('unique values :', df['Churn'].unique())
print('null count :', df['Churn'].isnull().sum())
print('value counts :')
print(df['Churn'].value_counts().to_string()) # Count of churned vs retained customers

Churn Column :

unique values : ['No' 'Yes']
null count : 0
value counts :
Churn
No     5174
Yes    1869


## 4. Duplicate CustomerID Check

In [25]:
duplicate_count = df['customerID'].duplicated().sum()
print(f'duplicate CustomerIDs found : {duplicate_count}')

duplicate CustomerIDs found : 0


## 5. Fixing column data types

In [26]:
# Investigate the blank rows before touching anything

blank_mask = df['TotalCharges'].str.strip() == ''

print(f'Blank TotalCharges rows : {blank_mask.sum()}')
print(f'Their tenure values     : {df.loc[blank_mask, "tenure"].values}')
print(f'Their Churn values      : {list(df.loc[blank_mask, "Churn"].values)}')

Blank TotalCharges rows : 11
Their tenure values     : [0 0 0 0 0 0 0 0 0 0 0]
Their Churn values      : ['No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No']


In [27]:
# Fixing the column

df['TotalCharges'] = df['TotalCharges'].str.strip()

df['TotalCharges'] = df['TotalCharges'].replace('', np.nan)

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])

df['TotalCharges'] = df['TotalCharges'].fillna(0.0)

# Verify the fix
print(f'dtype after fix : {df["TotalCharges"].dtype}')
print(f'Null values left : {df["TotalCharges"].isnull().sum()}')
print(f'Min value : {df["TotalCharges"].min()}')
print(f'Max value : ${df["TotalCharges"].max():,.2f}')

dtype after fix : float64
Null values left : 0
Min value : 0.0
Max value : $8,684.80


In [28]:
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

print('SeniorCitizen distribution after conversion:')
print(df['SeniorCitizen'].value_counts().to_string())

senior_pct = (df['SeniorCitizen'] == 'Yes').sum() / len(df) * 100
print(f'\n→ Senior citizens make up {senior_pct:.1f}% of the customer base')

SeniorCitizen distribution after conversion:
SeniorCitizen
No     5901
Yes    1142

→ Senior citizens make up 16.2% of the customer base


## 6. Feature Engineering

In [29]:
# Lets make bucket boundaries for 'tenure'
bins   = [0, 12, 24, 48, 72]
labels = ['0-12 Months', '13-24 Months', '25-48 Months', '49-72 Months']

df['tenure_group'] = pd.cut(
    df['tenure'],           
    bins=bins,             
    labels=labels,          
    include_lowest=True  
)

print('tenure_group distribution:')
tg_dist = df['tenure_group'].value_counts().sort_index()
for group, count in tg_dist.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)  # Simple text bar chart
    print(f'  {str(group):<20} : {count:>5,} customers ({pct:.1f}%)  {bar}')

tenure_group distribution:
  0-12 Months          : 2,186 customers (31.0%)  ███████████████
  13-24 Months         : 1,024 customers (14.5%)  ███████
  25-48 Months         : 1,594 customers (22.6%)  ███████████
  49-72 Months         : 2,239 customers (31.8%)  ███████████████


In [30]:
# Columns where Yes/No has direct business meaning for encoding
binary_cols = [
    'SeniorCitizen',     # We have converted binary to string(Yes/No) in prev cell
    'Partner',           # Has a partner: Yes/No
    'Dependents',        # Has dependents: Yes/No
    'PhoneService',      # Has phone service: Yes/No
    'PaperlessBilling',  # Enrolled in paperless billing: Yes/No
    'Churn'              # Target variable: Yes = churned, No = retained
]

for col in binary_cols:
    df[col + '_enc'] = df[col].map({'Yes': 1, 'No': 0})

print('Encoded columns created:')
print(f'{"Column":<28} {"Count = 1":>10}  {"% of base":>10}')
print('-' * 52)
for col in binary_cols:
    enc = col + '_enc'
    ones = df[enc].sum()
    pct  = ones / len(df) * 100
    print(f'{enc:<28} {ones:>10,}  {pct:>9.1f}%')

Encoded columns created:
Column                        Count = 1   % of base
----------------------------------------------------
SeniorCitizen_enc                 1,142       16.2%
Partner_enc                       3,402       48.3%
Dependents_enc                    2,110       30.0%
PhoneService_enc                  6,361       90.3%
PaperlessBilling_enc              4,171       59.2%
Churn_enc                         1,869       26.5%


## 7. Final Data Validation

In [31]:
# Null check across all columns
print('=== NULL CHECK ===')
null_counts = df.isnull().sum()
cols_with_nulls = null_counts[null_counts > 0]

if len(cols_with_nulls) == 0:
    print('Zero nulls in all columns. Dataset is clean.')
else:
    print('Columns with nulls — investigate before proceeding:')
    print(cols_with_nulls)

=== NULL CHECK ===
Zero nulls in all columns. Dataset is clean.


In [32]:
# Final shape
print('=== FINAL DATASET SHAPE ===')
print(f'Rows : {df.shape[0]:,}')
print(f'Columns : {df.shape[1]}')
print(f'Original cols : 21')
print(f'New cols added : {df.shape[1] - 21}  (tenure_group + 6 _enc columns)')

print('\nAll columns:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:>2}. {col:<30} [{df[col].dtype}]')

=== FINAL DATASET SHAPE ===
Rows : 7,043
Columns : 28
Original cols : 21
New cols added : 7  (tenure_group + 6 _enc columns)

All columns:
   1. customerID                     [object]
   2. gender                         [object]
   3. SeniorCitizen                  [object]
   4. Partner                        [object]
   5. Dependents                     [object]
   6. tenure                         [int64]
   7. PhoneService                   [object]
   8. MultipleLines                  [object]
   9. InternetService                [object]
  10. OnlineSecurity                 [object]
  11. OnlineBackup                   [object]
  12. DeviceProtection               [object]
  13. TechSupport                    [object]
  14. StreamingTV                    [object]
  15. StreamingMovies                [object]
  16. Contract                       [object]
  17. PaperlessBilling               [object]
  18. PaymentMethod                  [object]
  19. MonthlyCharges              

In [33]:
# Business metrics sanity check
# Re-confirm key numbers — if these don't match our earlier check, something went wrong
churned = df[df['Churn'] == 'Yes']

print('=== BUSINESS METRICS SANITY CHECK ===')
print(f'Total customers       : {len(df):,}')
print(f'Churned               : {len(churned):,}')
print(f'Retained              : {len(df) - len(churned):,}')
print(f'Churn Rate            : {len(churned)/len(df)*100:.2f}%')
print(f'Avg Monthly Charge    : ${df["MonthlyCharges"].mean():.2f}')
print(f'Avg Charge (Churned)  : ${churned["MonthlyCharges"].mean():.2f}')
print(f'Avg Charge (Retained) : ${df[df["Churn"]=="No"]["MonthlyCharges"].mean():.2f}')
print(f'Monthly Revenue Lost  : ${churned["MonthlyCharges"].sum():,.2f}')
print(f'Annual Revenue Lost   : ${churned["MonthlyCharges"].sum()*12:,.2f}')

=== BUSINESS METRICS SANITY CHECK ===
Total customers       : 7,043
Churned               : 1,869
Retained              : 5,174
Churn Rate            : 26.54%
Avg Monthly Charge    : $64.76
Avg Charge (Churned)  : $74.44
Avg Charge (Retained) : $61.27
Monthly Revenue Lost  : $139,130.85
Annual Revenue Lost   : $1,669,570.20


## 8. Save the Cleaned Dataset

In [34]:
df.to_csv('data/telco_churn_cleaned.csv', index=False)

print(f'Cleaned dataset saved to: data')

Cleaned dataset saved to: data
